In [1]:
import os
import shutil
import numpy as np
import torchaudio
from pathlib import Path
from tqdm import tqdm

# --- Configuration ---
SOURCE_NOISY = Path("data/train/noisy_trainset_28spk_wav")
SOURCE_CLEAN = Path("data/train/clean_trainset_28spk_wav")
DEST_NOISY = Path("data/train/representative_noisy")
DEST_CLEAN = Path("data/train/representative_clean")

# Set to None to copy ALL files. Set to a number (e.g., 1000) for a stratified miniature subset.
SUBSET_SIZE = 500             

N_DENSITY_BINS = 5               # Stratify by Speech Density (e.g., 0-20%, 20-40%, etc.)
N_DURATION_BINS = 4              # Stratify by Audio Duration
SILENCE_THRESHOLD_DB = -40
TARGET_SR = 16000

def analyze_audio(file_path):
    """Extracts both speech density and duration for 2D stratification."""
    try:
        wave, sr = torchaudio.load(str(file_path))
        if wave.shape[0] > 1:
            wave = wave.mean(dim=0, keepdim=True)
        if sr != TARGET_SR:
            wave = torchaudio.transforms.Resample(sr, TARGET_SR)(wave)
            
        wave_np = wave.squeeze(0).numpy().astype(np.float32)
        duration = len(wave_np) / TARGET_SR
        
        if len(wave_np) == 0:
            return 0.0, 0.0
            
        frame_size = int(TARGET_SR * 0.010) 
        n_frames = len(wave_np) // frame_size
        
        if n_frames == 0:
            return 0.0, duration
            
        active_frames = 0
        for i in range(n_frames):
            frame = wave_np[i * frame_size : (i + 1) * frame_size]
            rms = np.sqrt(np.mean(frame ** 2)) + 1e-8
            db = 20 * np.log10(rms)
            if db > SILENCE_THRESHOLD_DB:
                active_frames += 1
                
        density = active_frames / n_frames
        return density, duration

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")
        return None, None

def main():
    DEST_NOISY.mkdir(parents=True, exist_ok=True)
    DEST_CLEAN.mkdir(parents=True, exist_ok=True)

    print("Scanning for valid audio pairs...")
    clean_files = sorted(list(SOURCE_CLEAN.glob('*.wav')))
    
    valid_pairs = []
    for c_file in clean_files:
        n_file = SOURCE_NOISY / c_file.name
        if n_file.exists():
            valid_pairs.append(c_file)
            
    print(f"Found {len(valid_pairs)} valid pairs. Analyzing 2D distribution...")
    
    # 1. Analyze all files
    data = []
    for c_file in tqdm(valid_pairs, desc="Analyzing audio"):
        density, duration = analyze_audio(c_file)
        if density is not None:
            data.append((c_file, density, duration))
            
    # 2. Create 2D Bins
    densities = [d[1] for d in data]
    durations = [d[2] for d in data]
    
    density_edges = np.linspace(0, 1.0, N_DENSITY_BINS + 1)
    duration_edges = np.linspace(min(durations), max(durations) + 0.01, N_DURATION_BINS + 1)
    
    bins = {}
    for file, density, duration in data:
        d_bin = np.searchsorted(density_edges, density, side='right') - 1
        dur_bin = np.searchsorted(duration_edges, duration, side='right') - 1
        
        d_bin = min(d_bin, N_DENSITY_BINS - 1)
        dur_bin = min(dur_bin, N_DURATION_BINS - 1)
        
        bin_key = (d_bin, dur_bin)
        if bin_key not in bins:
            bins[bin_key] = []
        bins[bin_key].append(file)

    # 3. Print 2D Matrix Picture of the FULL population
    print("\n" + "="*70)
    print("📊 FULL POPULATION DISTRIBUTION (File Counts)")
    print("Rows: Speech Density | Columns: Audio Duration")
    print("-" * 70)
    
    dur_labels = [f"{duration_edges[i]:.1f}-{duration_edges[i+1]:.1f}s" for i in range(N_DURATION_BINS)]
    print(f"{'Density':<12} | " + " | ".join([f"{l:<10}" for l in dur_labels]))
    print("-" * 70)
    
    for d in range(N_DENSITY_BINS):
        row_label = f"{int(density_edges[d]*100)}-{int(density_edges[d+1]*100)}%"
        row_data = []
        for dur in range(N_DURATION_BINS):
            count = len(bins.get((d, dur), []))
            row_data.append(f"{count:<10}")
        print(f"{row_label:<12} | " + " | ".join(row_data))
    print("="*70)

    # 4. Determine Sample Size
    total_files = len(data)
    target_size = total_files if SUBSET_SIZE is None else SUBSET_SIZE
    print(f"\nTarget subset size: {target_size} files")

    selected_files = []
    
    # 5. Sample from inside each 2D bin proportionally
    print("\nSelecting files proportionally from each bin...")
    for bin_key, files_in_bin in bins.items():
        proportion = len(files_in_bin) / total_files
        n_to_select = int(round(proportion * target_size))
        
        # Ensure rare bins get at least 1 file if they exist
        if n_to_select == 0 and len(files_in_bin) > 0 and target_size < total_files:
            n_to_select = 1
            
        n_to_select = min(n_to_select, len(files_in_bin))
        
        if n_to_select > 0:
            chosen_indices = np.random.choice(len(files_in_bin), n_to_select, replace=False)
            for idx in chosen_indices:
                selected_files.append(files_in_bin[idx])

    # 6. Copy the selected files
    print(f"\nCopying {len(selected_files)} stratified files to destination...")
    for c_file in tqdm(selected_files, desc="Copying files"):
        n_file = SOURCE_NOISY / c_file.name
        shutil.copy(n_file, DEST_NOISY / n_file.name)
        shutil.copy(c_file, DEST_CLEAN / c_file.name)
        
    print("\n✅ Stratified subset creation complete!")


In [2]:
if __name__ == "__main__":
    main()

Scanning for valid audio pairs...
Found 11572 valid pairs. Analyzing 2D distribution...


Analyzing audio: 100%|██████████| 11572/11572 [02:50<00:00, 67.86it/s]



📊 FULL POPULATION DISTRIBUTION (File Counts)
Rows: Speech Density | Columns: Audio Duration
----------------------------------------------------------------------
Density      | 1.1-4.6s   | 4.6-8.1s   | 8.1-11.6s  | 11.6-15.1s
----------------------------------------------------------------------
0-20%        | 2          | 0          | 0          | 1         
20-40%       | 721        | 2          | 0          | 0         
40-60%       | 6235       | 138        | 10         | 0         
60-80%       | 3611       | 387        | 40         | 9         
80-100%      | 387        | 29         | 0          | 0         

Target subset size: 500 files

Selecting files proportionally from each bin...

Copying 504 stratified files to destination...


Copying files: 100%|██████████| 504/504 [00:03<00:00, 134.82it/s]


✅ Stratified subset creation complete!
